# SDP Causality Extraction

The **Shortest Dependency Path (SDP)** approach extracts causal relations from text using a
syntactic dependency parse rather than fine-tuned neural weights.  Given a sentence, a
dependency parser produces a tree; the path connecting two entity spans in that tree tends to
contain the word or phrase that expresses their causal relationship.

This notebook evaluates the rule-based SDP baseline on two causalatee tasks:

| Task | Approach | Metric |
|------|----------|--------|
| [Causal Event Candidate Extraction](../tasks/causal_event_candidate_detection.md) | Subject/object of causal-verb constructions | Span F1 |
| [Causality Identification](../tasks/causality_identification.md) | Causal-lexicon lookup on the SDP between entity heads | Binary F1 |

See the [SDP model page](../models/sdp_causality_extraction.md) for a description of the
algorithm and its design decisions.

## Setup

Install spaCy, download the English model, and install causalatee's HuggingFace extras.

In [ ]:
%pip install -q causalatee[baselines]
!python -m spacy download en_core_web_sm -q

## Shared utilities

We need three building blocks used by both tasks:

1. **Dependency graph** — a `networkx` undirected graph over token indices.
2. **Span head** — the syntactic head of a character span (the token whose governor lies
   outside the span).
3. **Shortest dependency path (SDP)** — the shortest path between two head tokens in the
   undirected graph.

In [ ]:
import re
import networkx as nx
import spacy

nlp = spacy.load("en_core_web_sm")

def build_graph(doc):
    G = nx.Graph()
    for t in doc:
        G.add_node(t.i)
        if t.head != t:
            G.add_edge(t.i, t.head.i)
    return G

def span_head(doc, char_start, char_end):
    """Return the syntactic head of the character span [char_start, char_end)."""
    toks = [t for t in doc if t.idx >= char_start and t.idx + len(t.text) <= char_end]
    if not toks:
        toks = [t for t in doc if t.idx < char_end and t.idx + len(t.text) > char_start]
    if not toks:
        return None
    for t in toks:
        if t.head not in toks or t.head == t:
            return t
    return toks[0]

def get_sdp(doc, G, h1, h2):
    """Return the list of tokens on the shortest dependency path between h1 and h2."""
    try:
        return [doc[i] for i in nx.shortest_path(G, h1.i, h2.i)]
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return []

def subtree_span(token):
    """Return the (char_start, char_end) of the syntactic subtree rooted at token."""
    subtree = list(token.subtree)
    return (min(t.idx for t in subtree), max(t.idx + len(t.text) for t in subtree))

## Task 1: Causality Identification

### Dataset

The `causality identification` configuration wraps each entity pair in `<e1>…</e1>` and
`<e2>…</e2>` markers embedded in the sentence text.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("thagen/AltLex", "causality identification")
print(dataset)

Expected output:
```
DatasetDict({
    train: Dataset({features: ['text', 'relations'], num_rows: 1984})
    test:  Dataset({features: ['text', 'relations'], num_rows: 496})
})
```

### Rule-based SDP classifier

We first strip the entity markers from the text (recording where each span landed), then
find the SDP between the two span heads and check whether any token on the path belongs
to a curated **causal lexicon**.

In [ ]:
CAUSAL_LEMMAS = {
    # verbs
    "cause", "lead", "result", "trigger", "produce", "create", "generate",
    "induce", "bring", "make", "force", "enable", "prevent", "stop", "drive",
    "determine", "contribute", "allow", "permit", "provoke", "prompt",
    "mean", "explain", "ensure", "motivate", "necessitate", "require",
    "yield", "entail", "compel", "follow", "stem", "arise", "come",
    # nouns
    "cause", "reason", "result", "effect", "consequence", "outcome",
    "impact", "source", "origin", "factor", "basis",
    # prepositions / subordinators
    "because", "since", "due", "owing", "thanks", "through", "via",
}

def strip_markers(text):
    """Remove entity markers and return (clean_text, {entity_id: (start, end)})."""
    offsets, starts = {}, {}
    result, pos = "", 0
    for m in re.finditer(r"<(/?e\d+)>", text):
        result += text[pos:m.start()]
        tag = m.group(1)
        if not tag.startswith("/"):
            starts[tag] = len(result)
        else:
            eid = tag[1:]
            offsets[eid] = (starts[eid], len(result))
        pos = m.end()
    result += text[pos:]
    return result, offsets

def predict_identification(example):
    """Return (true_label, predicted_label) for one identification example."""
    rels = example.get("relations", [])
    true_label = int(any(r["relationship"] == 1 for r in rels))

    clean, spans = strip_markers(example["text"])
    s1, s2 = spans.get("e1"), spans.get("e2")
    if not s1 or not s2:
        return true_label, 0          # no entity markers → predict no-relation

    doc = nlp(clean)
    G = build_graph(doc)
    h1 = span_head(doc, *s1)
    h2 = span_head(doc, *s2)
    if not h1 or not h2:
        return true_label, 0

    path = get_sdp(doc, G, h1, h2)
    causal = any(t.lemma_.lower() in CAUSAL_LEMMAS for t in path)
    return true_label, int(causal)

### Evaluate

In [ ]:
import evaluate
import numpy as np

labels, preds = [], []
for ex in dataset["test"]:
    true, pred = predict_identification(ex)
    labels.append(true)
    preds.append(pred)

f1_metric = evaluate.load("f1")
results = f1_metric.compute(predictions=preds, references=labels, average="binary")
print(results)

Expected output:
```
{'f1': 0.6310}
```

The rule fires with **perfect precision** (every path that contains a canonical causal lemma
is indeed causal) but limited **recall** (0.461): AltLex is specifically a corpus of
*alternative* lexicalizations — causal connectives such as *meaning*, *in response to*,
*following* — that are not covered by the canonical lexicon.  A trained feature-based
classifier over the full path encoding, such as the SDP-SVM of [@rink:2010], would recover
more of these cases.

## Task 2: Causal Event Candidate Extraction

For candidate extraction no entity spans are pre-given.  The heuristic identifies tokens
whose lemma belongs to the causal lexicon and extracts their syntactic arguments
(subject, object, complement) as candidate cause/effect spans.

In [ ]:
from datasets import load_dataset as _ld

dataset_ex = _ld("thagen/AltLex", "causal candidate extraction")
print(dataset_ex)

Expected output:
```
DatasetDict({
    train: Dataset({features: ['text', 'entity'], num_rows: 1984})
    test:  Dataset({features: ['text', 'entity'], num_rows: 496})
})
```

In [ ]:
def extract_candidates(text):
    """
    Return a list of [char_start, char_end] candidate spans extracted via dependency parsing.

    Strategy:
      - For causal verbs: extract nsubj and obj/dobj subtrees.
      - For causal nouns: extract genitive and prepositional modifiers.
      - For causal prepositions/subordinators: extract the pobj subtree and the
        main-clause subject of the governing verb.
    """
    doc = nlp(text)
    spans = []
    for t in doc:
        if t.lemma_.lower() not in CAUSAL_LEMMAS:
            continue
        if t.pos_ == "VERB":
            for child in t.children:
                if child.dep_ in ("nsubj", "nsubjpass", "obj", "dobj", "ccomp", "attr"):
                    spans.append(list(subtree_span(child)))
        elif t.pos_ == "NOUN":
            for child in t.children:
                if child.dep_ in ("nsubj", "prep", "poss", "nmod"):
                    spans.append(list(subtree_span(child)))
        elif t.pos_ in ("ADP", "SCONJ", "ADV"):
            for child in t.children:
                spans.append(list(subtree_span(child)))
            if t.head.pos_ == "VERB":
                for sib in t.head.children:
                    if sib.dep_ in ("nsubj", "nsubjpass"):
                        spans.append(list(subtree_span(sib)))
    seen, result = set(), []
    for sp in spans:
        k = tuple(sp)
        if k not in seen:
            seen.add(k)
            result.append(sp)
    return result

### Evaluate

In [ ]:
from ctk.evaluation._spans import dataset_span_scores

all_preds, all_truths = [], []
for ex in dataset_ex["test"]:
    all_preds.append(extract_candidates(ex["text"]))
    all_truths.append(ex["entity"])

scores = dataset_span_scores(all_truths, all_preds)
print(scores)

Expected output:
```
{'precision': 0.1266, 'recall': 0.0468, 'f1': 0.0632,
 'granularity': 0.1757, 'f1_gran': 0.0583,
 'intersection_over_union': 0.0386}
```

The low scores reflect two compounding limitations on AltLex:

1. **Lexicon coverage** — AltLex uses *alternative* causal connectives; many do not lemmatise
   to a word in `CAUSAL_LEMMAS`, so the causal token is never identified and no spans are
   extracted.
2. **Argument structure mismatch** — even when the causal verb is found, the gold spans often
   include discourse-level spans (entire clauses) that differ from the syntactic subtrees
   extracted here.

On corpora with canonical causal verbs (*cause*, *lead to*, *result in*) such as
BECauSEv2 or CNC, the same heuristic achieves substantially higher recall.